## Image Representations

<h2><b>What is the meaning of 'representations of images'?</b></h2>

An image representation refers to the process of translating raw, high-dimensional visual data (like tens of thousands of RGB pixel values) into a compressed, abstract, or more useful mathematical form.
Fundamentally, image representation is rooted in dimensionality reduction. While raw image data $x$ exists in a massive, high-dimensional coordinate system, realistic images (like faces or natural scenes) only occupy a tiny subset or "manifold" of that space. Creating a representation means finding a "low-dimensional (or hidden) representation $h$, which can approximately explain the data $x$, so that $x \approx f[h,\theta]$".

<h3>What is achieved by image representation?</h3>

1. <i>Dimensionality reduction</i> : From thousands of rows of RGB values, we come down to few hundreds of rows of representation.
2. <i> A Compact Summary of Critical Information</i>: A representation serves as a "highly compact representation" that summarizes an image region while retaining the "critical information about the image appearance and layout".
3. <i> A Method to Filter Out Irrelevant Variations </i>: In real-world computer vision, raw RGB values constantly fluctuate due to factors that have nothing to do with the actual object, such as changes in ambient lighting, camera gain, or slight shifts in pose.
4. <i> An Intermediate Stepping Stone for Inference </i>: Representations act as an "intermediate representation for inferring higher-level properties".

<h2><b>The Dataset: ImageNet</b></h2>

**ImageNet** is a massive, foundational visual database designed specifically to support visual object recognition and computer vision research. It serves as a comprehensive "visual dictionary" for computers and has been instrumental in advancing deep learning models.

* **Scale and Scope:** The full dataset (sometimes referred to as ImageNet-21K) contains more than 14 million images that have been hand-annotated.
* **Organization:** It is structured according to the WordNet lexical hierarchy, where each node or concept (called a "synset") is depicted by hundreds or thousands of images. The full dataset contains over 20,000 distinct categories, ranging from specific dog breeds to everyday objects like balloons or strawberries.
* **Annotation:** The images were scraped from search engines and then rigorously human-annotated using Amazon's Mechanical Turk platform to verify the presence of specific objects. Over one million of these images also include object-level bounding boxes to show exactly where the object is located.

Working on entire ImageNet dataset is both laborious and unnecessary. Therefore, this study of creating representations is done on a subset of ImageNet. A 100,000 images were sampled at random from the dataset and were subsequently split into training, validation and test sets in the ratio of 8:1:1 respectively.

<h2><b>The Architecture: Convolutional Autoencoder</b></h2>

The ImageNet dataset is vast and it varies greatly. Hence the ideal architecture to encode the image is a **Convolutional Autoencoder** where stacks of <code>nn.Conv2d</code> and <code>nn.MaxPool2d</code> layers. Each step will increase the number of feature channels while slicing the spatial dimensions ($H \times W$) in half, squeezing the image down into your dense, low-dimensional latent representation.

The same network can later function as a decoder to extract images from the representations.

<h3><b>The Pipeline</b></h3>

* **1. Dataset:** 100k random ImageNet images, split 8:1:1 (80k Train, 10k Val, 10k Test).
* **2. Pre-processing:** Resize all images to a uniform resolution (e.g., $64 \times 64$), convert to PyTorch tensors, and normalize the RGB channels.
* **3. Architecture:** An hourglass design. The **Encoder** (`Conv2d` + `MaxPool`) compresses images into a dense latent representation. The **Decoder** (`ConvTranspose2d`) reconstructs the representation back into pixels.
* **4. Training & Validation:** Optimize using **MSE Loss** (measuring pixel reconstruction error). Monitor validation loss after each epoch to trigger **Early Stopping** and prevent overfitting.



## <b>Data Loading</b>


In [17]:
import numpy as np
import matplotlib.pyplot as plt
import torch

# setting up seed
seed = 42
torch.manual_seed(seed=seed)
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(42)

# 4. Force PyTorch to use deterministic algorithms
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Random seeds locked to {seed}. Training is now reproducible.")

Random seeds locked to 42. Training is now reproducible.


In [19]:
# set up the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Subset, dataloader

In [ ]:
transform = transforms(
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.486, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

from datasets import load_dataset

ds = load_dataset("zh-plus/tiny-imagenet")

